In [20]:
import pandas as pd
import numpy as np

import requests
import re
from bs4 import BeautifulSoup
import difflib
from tqdm import tqdm
import os
import time

Preparation: Get paths to the pdfs

In [21]:
path_constant_part = "C:/Users/jonas/Documents/Studium - PhD Informatik/Survey Paper Interaction/Study PDFs"

pdf_path_dict = {
    1: '33_weisenberger.pdf',
    2: '5019_brewster.pdf',
    3: '32_metzger.pdf',
    4: '40_buil.pdf',
    5: '84_buil.pdf',
    6: '61_manabe.pdf',
    7: '5011_simpson.pdf',
    8: '184_tamaki.pdf',
    9: '62_sano.pdf',
    10: '5012_simpson.pdf',
    11: '253_gamper.pdf',
    12: '1431_jeong.pdf',
    13: '23_manabe.pdf',
    14: '85_matsumura.pdf',
    15: '94_tessendorf.pdf',
    16: '1508_tessendorf.pdf',
    17: '1539_tessendorf.pdf',
    18: '25_akiyama.pdf',
    19: '20_manabe.pdf',
    20: '1_lissermann.pdf',
    21: '93_sahni.pdf',
    22: '7_bedri.pdf',
    23: '190_bleichner.pdf',
    24: '92_norton.pdf',
    25: '188_wang.pdf',
    26: '19_weigel.pdf',
    27: '63_ashbrook.pdf',
    28: '5021_curran.pdf',
    29: '34_laput.pdf',
    30: '69_merrill.pdf',
    31: '43_nguyen.pdf',
    32: '17_ando.pdf',
    33: '206_dim.pdf',
    34: '5023_favre_felix.pdf',
    35: '22_kikuchi.pdf',
    36: '64_maag.pdf',
    37: '12_matthies.pdf',
    38: '111_wang.pdf',
    39: '117_ahn.pdf',
    40: '177_carioli.pdf',
    41: '91_favre_felix.pdf',
    42: '31_huang.pdf',
    43: '5020_cheah.pdf',
    44: '18_lee.pdf',
    45: '5001_min.pdf',
    46: '65_nguyen.pdf',
    47: '60_taniguchi.pdf',
    48: '8_amesaka.pdf',
    49: '80_ferlini.pdf',
    50: '77_hoelzemann.pdf',
    51: '106_lee.pdf',
    52: '14_lee.pdf',
    53: '2_nasser.pdf',
    54: '78_odoemelem.pdf',
    55: '15_shirota.pdf',
    56: '5008_vega_galvez.pdf',
    57: '27_yan.pdf',
    58: '223_cao.pdf',
    59: '234_chen.pdf',
    60: '212_chen.pdf',
    61: '149_kaveh.pdf',
    62: '16_prakash.pdf',
    63: '26_xu.pdf',
    64: '5002_yang.pdf',
    65: '229_fan.pdf',
    66: '247_ferlini.pdf',
    67: '258_gashi.pdf',
    68: '217_hashem.pdf',
    69: '5010_islam.pdf',
    70: '226_jin.pdf',
    71: '5006_kakaraparthi.pdf',
    72: '213_khanna.pdf',
    73: '224_laporte.pdf',
    74: '5021_ma.pdf',
    75: '259_nasser.pdf',
    76: '5015_peng.pdf',
    77: '227_röddiger.pdf',
    78: '221_sun.pdf',
    79: '231_verma.pdf',
    80: '5004_wu.pdf',
    81: '4066_alkiek.pdf',
    82: '5003_bi.pdf',
    83: '4002_bi.pdf',
    84: '3017_choi.pdf',
    85: '5014_futami.pdf',
    86: '3016_jin.pdf',
    87: '3013_li.pdf',
    88: '3028_rateau.pdf',
    89: '4003_song.pdf',
    90: '3021_srivastava.pdf',
    91: '3007_wang.pdf',
    92: '3015_wang.pdf',
    93: '4068_yang.pdf',
    94: '3067_alkiek.pdf',
    95: '3054_chugh.pdf',
    96: '3208_iguma.pdf',
    97: '3058_jin.pdf',
    98: '3043_li.pdf',
    99: '3062_panda.pdf',
    100: '4105_paul.pdf',
    101: '3063_stanke.pdf',
    102: '4090_yi.pdf',
    103: '5007_zhang.pdf',
    104: '3083_zhang.pdf',
    105: '3027_zhang.pdf',
    106: '3104_dong.pdf',
    107: '4188_ge.pdf',
    108: '4189_hu.pdf',
    109: '4194_hu.pdf',
    110: '3122_lepold.pdf',
    111: '4220_ronco.pdf',
    112: '3119_sato.pdf',
    113: '3096_shimon.pdf',
    114: '3098_shojaeifard.pdf',
    115: '3139_srivastava.pdf',
    116: '3142_srivastava.pdf',
    117: '4195_sun.pdf',
    118: '3134_suzuki.pdf',
    119: '3097_wang.pdf',
    120: '4208_wang.pdf',
    121: '4238_wang.pdf',
    122: '4237_xie.pdf',
    123: '3105_yang.pdf',
    124: '4192_yi.pdf',
    125: '3156_amesaka.pdf',
    126: '4300_chugh.pdf',
    127: '3189_gao.pdf',
    128: '3171_kang.pdf',
    129: '3154_kim.pdf',
    130: '3164_liu.pdf',
    131: '4333_tang.pdf',
    132: '3162_yang.pdf',
    133: '3158_zhao.pdf',
    134: '4303_zhu.pdf',
    135: '3172_zitz.pdf',
    136: '2074_bodsworth.pdf',
    137: '3230_dong.pdf',
    138: '4413_liu.pdf',
    139: '4425_takaki.pdf',
    140: '3228_van_oort.pdf',
    141: '3193_yuma.pdf',
}

pdf_path_dict_keys = list(pdf_path_dict.keys())

In [22]:
# Dictionary to track valid and invalid paths
path_status = {
    'valid': [],
    'invalid': []
}

# Loop through all entries in pdf_path_dict to check if files exist
print("Checking PDF file paths...")
for id_num, pdf_name in tqdm(pdf_path_dict.items()):
    full_path = os.path.join(path_constant_part, pdf_name)
    
    if os.path.isfile(full_path):
        path_status['valid'].append((id_num, pdf_name))
    else:
        path_status['invalid'].append((id_num, pdf_name))

# Print results
print(f"\nResults:")
print(f"- Valid paths: {len(path_status['valid'])}/{len(pdf_path_dict)}")
print(f"- Invalid paths: {len(path_status['invalid'])}/{len(pdf_path_dict)}")

# Print details of invalid paths if any exist
if path_status['invalid']:
    print("\nInvalid paths:")
    for id_num, pdf_name in path_status['invalid']:
        print(f"ID {id_num}: {path_constant_part}{pdf_name}")

Checking PDF file paths...


100%|██████████| 141/141 [00:00<00:00, 10491.71it/s]


Results:
- Valid paths: 141/141
- Invalid paths: 0/141


Get the Citation Matrix of out of the PDFs

In [23]:
# DF with ID and bibtex
df_id_bibtex = pd.read_excel('../datasets/interconnections/bibtex_mapping_of_ids.xlsx')

In [24]:

# Helper: normalize DOI to bare form for reliable comparison
def normalize_doi(doi: str) -> str:
    """Strip URL prefixes from DOIs and lowercase so different source formats compare equal."""
    doi = doi.lower().strip()
    for prefix in [
        'https://doi.org/',
        'http://doi.org/',
        'https://dx.doi.org/',
        'http://dx.doi.org/',
        'doi:',
    ]:
        if doi.startswith(prefix):
            doi = doi[len(prefix):]
            break
    return doi


In [25]:
# Function to extract references from a paper using GROBID and convert to BibTeX
def extract_references_to_bibtex(pdf_path):
    """Extract references from PDF using GROBID and convert to BibTeX format"""
    references = []
    
    try:
        # Read PDF content
        with open(pdf_path, "rb") as pdf_file:
            pdf_content = pdf_file.read()
        
        # Request GROBID to process references
        files = {"input": ("document.pdf", pdf_content, "application/pdf")}
        response = requests.post(
            "http://localhost:8070/api/processReferences", 
            files=files, 
            timeout=300
        )
        
        if response.status_code != 200:
            print(f"Error from GROBID API: {response.status_code}")
            return []
            
        # Parse XML response
        soup = BeautifulSoup(response.text, 'xml')
        
        for i, bibl in enumerate(soup.find_all('biblStruct')):
            # Extract key citation components
            ref_id = bibl.get('xml:id', f'ref_{i}')
            
            # Authors
            authors = []
            for author_tag in bibl.find_all('author'):
                person = author_tag.find('persName')
                if person:
                    surname = person.find('surname')
                    forename = person.find('forename')
                    
                    if surname:
                        author_name = surname.text
                        if forename:
                            author_name = f"{author_name}, {forename.text}"
                        authors.append(author_name)
            
            # Title
            title = ""
            title_tag = bibl.find('title', {'level': 'a'})
            if title_tag:
                title = title_tag.text.strip()
            
            # Year
            year = ""
            date_tag = bibl.find('date', {'type': 'published'})
            if date_tag and date_tag.get('when'):
                year = date_tag.get('when').split('-')[0]  # Extract year from date
            
            # Journal/Conference
            journal = ""
            journal_tag = bibl.find('title', {'level': 'j'})
            if journal_tag:
                journal = journal_tag.text.strip()
            else:
                book_tag = bibl.find('title', {'level': 'm'})
                if book_tag:
                    journal = book_tag.text.strip()
            
            # Volume, Issue, Pages
            volume = ""
            vol_tag = bibl.find('biblScope', {'unit': 'volume'})
            if vol_tag:
                volume = vol_tag.text.strip()
            
            issue = ""
            issue_tag = bibl.find('biblScope', {'unit': 'issue'})
            if issue_tag:
                issue = issue_tag.text.strip()
            
            pages = ""
            pages_from_tag = bibl.find('biblScope', {'unit': 'page', 'from': True})
            pages_to_tag = bibl.find('biblScope', {'unit': 'page', 'to': True})
            if pages_from_tag and pages_to_tag:
                pages = f"{pages_from_tag.get('from')}--{pages_to_tag.get('to')}"
            elif pages_from_tag:
                pages = pages_from_tag.get('from')
            
            # DOI
            doi = ""
            doi_tag = bibl.find('idno', {'type': 'DOI'})
            if doi_tag:
                doi = doi_tag.text.strip()
            
            # Create BibTeX entry
            bibtex_id = f"grobid_{ref_id.replace('b', '')}"
            bibtex_str = f"@article{{{bibtex_id},\n"
            
            if authors:
                bibtex_str += f"  author = {{{' and '.join(authors)}}},\n"
            if title:
                bibtex_str += f"  title = {{{title}}},\n"
            if journal:
                bibtex_str += f"  journal = {{{journal}}},\n"
            if year:
                bibtex_str += f"  year = {{{year}}},\n"
            if volume:
                bibtex_str += f"  volume = {{{volume}}},\n"
            if issue:
                bibtex_str += f"  number = {{{issue}}},\n"
            if pages:
                bibtex_str += f"  pages = {{{pages}}},\n"
            if doi:
                bibtex_str += f"  doi = {{{doi}}},\n"
                
            bibtex_str += "}"
            
            # Create structured citation data
            citation = {
                'bibtex': bibtex_str,
                'authors': authors,
                'title': title,
                'year': year,
                'journal': journal,
                'doi': doi,
                'raw_xml': str(bibl),
                'author_last_names': [a.split(',')[0].lower().strip() if ',' in a else a.split()[-1].lower().strip() 
                                      for a in authors]
            }
            
            references.append(citation)
            
    except Exception as e:
        print(f"Error extracting references: {str(e)}")
    
    return references

In [26]:

# Function to normalize BibTeX entries for comparison
def normalize_bibtex_for_comparison(bibtex_str):
    """Extract key information from BibTeX for comparison"""
    result = {
        'authors': [],
        'title': '',
        'year': '',
        'journal': '',
        'doi': '',
        'author_last_names': []
    }
    
    # Extract authors
    author_match = re.search(r'author\s*=\s*\{(.*?)\}', bibtex_str, re.DOTALL)
    if author_match:
        authors_str = author_match.group(1)
        authors = [a.strip() for a in authors_str.split(' and ')]
        result['authors'] = authors
        # Preserve order – first author position matters for matching
        result['author_last_names'] = [a.split(',')[0].lower().strip() if ',' in a 
                                      else a.split()[-1].lower().strip() 
                                      for a in authors]
    
    # Extract title (strip nested braces that BibTeX uses for case protection)
    title_match = re.search(r'title\s*=\s*\{(.*?)\}', bibtex_str, re.DOTALL)
    if title_match:
        result['title'] = re.sub(r'[\{\}]', '', title_match.group(1).lower())
    
    # Extract year
    year_match = re.search(r'year\s*=\s*\{?(\d{4})\}?', bibtex_str)
    if year_match:
        result['year'] = year_match.group(1)
    
    # Extract journal or booktitle
    journal_match = re.search(r'journal\s*=\s*\{(.*?)\}', bibtex_str, re.DOTALL)
    if not journal_match:
        journal_match = re.search(r'booktitle\s*=\s*\{(.*?)\}', bibtex_str, re.DOTALL)
    if journal_match:
        result['journal'] = journal_match.group(1).lower()
    
    # Extract DOI – normalize to bare form for reliable comparison
    doi_match = re.search(r'doi\s*=\s*\{(.*?)\}', bibtex_str, re.DOTALL)
    if doi_match:
        result['doi'] = normalize_doi(doi_match.group(1))
    
    return result


In [27]:

# Function to match citation with corpus papers
def match_citation_to_corpus(citation_data, corpus_entries, citing_paper_id, threshold=0.5):
    """
    Match a citation to papers in the corpus.

    Temporal constraint: a paper can only be cited by papers published in the same
    year or later.  Corpus entries whose year > citing paper's year are skipped
    unconditionally, eliminating the need for post-hoc forward-citation correction.
    """
    matches = []

    # --- Citing paper's year for temporal guard ---
    citing_year_str = corpus_entries.get(citing_paper_id, {}).get('year', '')

    # --- Normalize citation fields ---
    citation_title = re.sub(r'[^\w\s]', '', citation_data.get('title', '').lower())
    citation_year  = citation_data.get('year', '')
    citation_authors = citation_data.get('author_last_names', [])   # ordered list
    citation_doi   = normalize_doi(citation_data.get('doi', ''))

    for paper_id, paper_data in corpus_entries.items():

        # Skip self-citations
        if paper_id == citing_paper_id:
            continue

        # ------------------------------------------------------------------
        # Temporal constraint: cited paper cannot be from a LATER year than
        # the citing paper.  Only enforced when both years are known.
        # ------------------------------------------------------------------
        cited_year_str = paper_data.get('year', '')
        if citing_year_str and cited_year_str:
            try:
                if int(cited_year_str) > int(citing_year_str):
                    continue
            except ValueError:
                pass  # malformed year – allow the candidate through

        score     = 0
        max_score = 0
        match_details = {}

        # --- Normalize corpus entry fields ---
        paper_title   = re.sub(r'[^\w\s]', '', paper_data.get('title', '').lower())
        paper_year    = paper_data.get('year', '')
        paper_authors = paper_data.get('author_last_names', [])   # ordered list
        paper_doi     = normalize_doi(paper_data.get('doi', ''))

        # ------------------------------------------------------------------
        # DOI match – highest confidence, already past temporal guard above
        # ------------------------------------------------------------------
        if citation_doi and paper_doi and citation_doi == paper_doi:
            return [{'paper_id': paper_id, 'score': 1.0, 'match_type': 'doi'}]

        # ------------------------------------------------------------------
        # Author matching (max 2 pts)
        # First author is a much stronger signal than co-authors.
        # ------------------------------------------------------------------
        if citation_authors and paper_authors:
            max_score += 2
            all_matching = set(citation_authors) & set(paper_authors)
            if all_matching:
                if citation_authors[0] == paper_authors[0]:
                    # First authors agree → strong signal (1.5 pts)
                    score += 1.5
                    match_details['first_author_match'] = True
                    # Any additional co-author overlap (up to 0.5 pts)
                    other_matches = all_matching - {citation_authors[0]}
                    score += min(0.5, len(other_matches) * 0.25)
                else:
                    # First authors differ, but some name overlap exists (1 pt max)
                    score += min(1.0, len(all_matching) * 0.5)
                match_details['matching_authors'] = list(all_matching)

        # ------------------------------------------------------------------
        # Year match (1 pt)
        # ------------------------------------------------------------------
        if citation_year and paper_year:
            max_score += 1
            if citation_year == paper_year:
                score += 1
                match_details['year_match'] = True

        # ------------------------------------------------------------------
        # Title similarity (max 5 pts)
        # SequenceMatcher handles substring/typo cases;
        # Jaccard word-overlap handles word-order differences (common in
        # abbreviated or reformatted titles extracted from PDFs).
        # ------------------------------------------------------------------
        if citation_title and paper_title:
            max_score += 5

            seq_sim = difflib.SequenceMatcher(None, citation_title, paper_title).ratio()

            c_words = set(citation_title.split())
            p_words = set(paper_title.split())
            jaccard = len(c_words & p_words) / len(c_words | p_words) if (c_words | p_words) else 0.0

            title_similarity = max(seq_sim, jaccard)

            # Containment bonus: one title is a substring of the other
            if len(citation_title) > 10 and len(paper_title) > 10:
                if citation_title in paper_title or paper_title in citation_title:
                    title_similarity = max(title_similarity, 0.85)

            score += title_similarity * 5
            match_details['title_similarity'] = round(title_similarity, 4)

        # ------------------------------------------------------------------
        # Guard against thin-data false positives:
        # require at least title + one other field to have contributed.
        # ------------------------------------------------------------------
        if max_score > 0:
            # Ensure at least 2 independent evidence types are present
            evidence_types = sum([
                bool(citation_authors and paper_authors),
                bool(citation_year and paper_year),
                bool(citation_title and paper_title),
            ])
            final_score = score / max_score
            if final_score >= threshold and evidence_types >= 2:
                matches.append({
                    'paper_id': paper_id,
                    'score': final_score,
                    'details': match_details
                })

    matches.sort(key=lambda x: x['score'], reverse=True)
    return matches


In [28]:

# Function to build citation network with Excel-based confirmation
def build_citation_network_with_excel(df_id_bibtex, pdf_path_dict, path_constant_part):
    """Build a citation network from papers and their references with Excel-based confirmation"""
    print("Preparing citation network analysis...")
    
    # Step 1: Create normalized corpus entries from BibTeX data
    print("Normalizing corpus BibTeX entries...")
    corpus_entries = {}
    
    for _, row in df_id_bibtex.iterrows():
        paper_id = row['ID']
        bibtex_str = row['Bibtex'] if isinstance(row['Bibtex'], str) else ""
        
        if bibtex_str:
            normalized_data = normalize_bibtex_for_comparison(bibtex_str)
            corpus_entries[paper_id] = normalized_data
    
    print(f"Normalized {len(corpus_entries)} papers in corpus")
    
    # Step 2: Initialize citation matrix
    paper_ids = list(corpus_entries.keys())
    citation_matrix = pd.DataFrame(0, index=paper_ids, columns=paper_ids)
    
    # Step 3: Extract all references first 
    print("First extracting all references from papers...")
    all_paper_references = {}
    
    # Create output directory for citations
    os.makedirs("../datasets/interconnections/extracted_citations", exist_ok=True)
    
    for paper_id in tqdm(corpus_entries.keys()):
        
        time.sleep(1)  # To avoid overwhelming the server
        
        if paper_id in pdf_path_dict:
            # Fix: use os.path.join to correctly build the full path
            pdf_path = os.path.join(path_constant_part, pdf_path_dict[paper_id])
            
            # Extract references
            print(f"\nExtracting references from paper {paper_id}...")
            references = extract_references_to_bibtex(pdf_path)
            print(f"Found {len(references)} references in paper {paper_id}")
            
            # Save references to file
            with open(f"../datasets/interconnections/extracted_citations/paper_{paper_id}_citations.bib", "w", encoding="utf-8") as f:
                for ref in references:
                    f.write(ref['bibtex'] + "\n\n")
            
            all_paper_references[paper_id] = references
    
    # Step 4: Match references to corpus and collect uncertain matches
    print("\nMatching references to corpus papers...")
    citation_details = {}
    uncertain_matches = []  # Store uncertain matches for Excel confirmation
    high_confidence_matches = []  # Store high confidence matches
    
    for citing_id, references in all_paper_references.items():
        citation_details[citing_id] = []
        
        for ref_idx, ref in enumerate(references):
            matches = match_citation_to_corpus(ref, corpus_entries, citing_id)
            
            if matches:
                best_match = matches[0]
                cited_id = best_match['paper_id']
                match_score = best_match['score']

                # Retrieve corpus paper info for review columns
                corpus_cited = corpus_entries.get(cited_id, {})
                corpus_cited_title   = corpus_cited.get('title', '')
                corpus_cited_authors = ', '.join(corpus_cited.get('author_last_names', []))
                corpus_cited_year    = corpus_cited.get('year', '')
                
                # High-confidence match (add directly)
                if match_score >= 0.7:
                    citation_matrix.loc[citing_id, cited_id] = 1
                    high_confidence_matches.append({
                        'citing_id': citing_id,
                        'cited_id': cited_id,
                        'score': match_score,
                        'citation_title': ref.get('title', ''),
                        'citation_authors': ', '.join(ref.get('authors', [])),
                        'citation_year': ref.get('year', ''),
                        'corpus_cited_title': corpus_cited_title,
                        'corpus_cited_authors': corpus_cited_authors,
                        'corpus_cited_year': corpus_cited_year,
                        'confidence': 'high',
                        'matching_authors': ', '.join(best_match.get('details', {}).get('matching_authors', [])),
                        'first_author_match': 'Yes' if best_match.get('details', {}).get('first_author_match', False) else 'No',
                        'title_similarity': best_match.get('details', {}).get('title_similarity', 0),
                        'year_match': 'Yes' if best_match.get('details', {}).get('year_match', False) else 'No',
                        'confirmed': 'Yes'  # Auto-confirmed due to high confidence
                    })
                    print(f"✓ Paper {citing_id} cites paper {cited_id} (score: {match_score:.2f})")
                
                # Uncertain match (store for Excel confirmation)
                elif match_score >= 0.5:
                    citing_filename = pdf_path_dict.get(citing_id, f"Unknown-{citing_id}")
                    cited_filename  = pdf_path_dict.get(cited_id,  f"Unknown-{cited_id}")
                    
                    uncertain_matches.append({
                        'citing_id': citing_id,
                        'citing_filename': citing_filename,
                        'cited_id': cited_id,
                        'cited_filename': cited_filename,
                        'score': match_score,
                        # --- Extracted reference fields ---
                        'citation_title': ref.get('title', ''),
                        'citation_authors': ', '.join(ref.get('authors', [])),
                        'citation_year': ref.get('year', ''),
                        # --- Corpus paper fields for side-by-side review ---
                        'corpus_cited_title': corpus_cited_title,
                        'corpus_cited_authors': corpus_cited_authors,
                        'corpus_cited_year': corpus_cited_year,
                        'confidence': 'medium',
                        'matching_authors': ', '.join(best_match.get('details', {}).get('matching_authors', [])),
                        'first_author_match': 'Yes' if best_match.get('details', {}).get('first_author_match', False) else 'No',
                        'title_similarity': best_match.get('details', {}).get('title_similarity', 0),
                        'year_match': 'Yes' if best_match.get('details', {}).get('year_match', False) else 'No',
                        'confirmed': ''  # To be filled in Excel
                    })
                    print(f"? Uncertain match: Paper {citing_id} possibly cites paper {cited_id} (score: {match_score:.2f})")
    
    # Step 5: Export uncertain matches to Excel
    if uncertain_matches:
        print(f"\nExporting {len(uncertain_matches)} uncertain matches to Excel...")
        uncertain_df = pd.DataFrame(uncertain_matches)
        
        # Add instructions in first row
        instructions = pd.DataFrame([{
            'citing_id': 'INSTRUCTIONS',
            'citing_filename': 'Fill in the "confirmed" column with: Yes, No, or leave blank to skip',
            'cited_id': '',
            'cited_filename': '',
            'score': '',
            'citation_title': 'Extracted from PDF by GROBID',
            'citation_authors': '',
            'citation_year': '',
            'corpus_cited_title': 'From your corpus BibTeX',
            'corpus_cited_authors': '',
            'corpus_cited_year': '',
            'confidence': '',
            'matching_authors': '',
            'first_author_match': '',
            'title_similarity': '',
            'year_match': '',
            'confirmed': ''
        }])
        
        # Combine instructions with data
        export_df = pd.concat([instructions, uncertain_df], ignore_index=True)
        
        # Export to Excel
        excel_path = '../datasets/interconnections/citation_confirmation.xlsx'
        export_df.to_excel(excel_path, index=False)
        print(f"Exported uncertain matches to {excel_path}")
        print("Please fill in the 'confirmed' column with 'Yes' or 'No' and save the file.")
        print("Then run the import_citation_confirmations() function to update the citation matrix.")
    
    # Also export high confidence matches for reference
    if high_confidence_matches:
        high_conf_df = pd.DataFrame(high_confidence_matches)
        high_conf_df.to_excel('../datasets/interconnections/high_confidence_citations.xlsx', index=False)
    
    return citation_matrix, citation_details, uncertain_matches


In [29]:
def import_citation_confirmations(citation_matrix):
    """Import citation confirmations from Excel and update the citation matrix"""
    confirmation_file = '../datasets/interconnections/citation_confirmation.xlsx'
    
    if not os.path.exists(confirmation_file):
        print(f"Error: Confirmation file {confirmation_file} not found.")
        return citation_matrix
    
    # Load confirmations, skipping the instruction row
    confirmations = pd.read_excel(confirmation_file, header=0, skiprows=[1])
        
    # Count statistics
    confirmed_count = 0
    rejected_count = 0
    skipped_count = 0
    
    # Process each confirmation
    for _, row in confirmations.iterrows():
        citing_id = row['citing_id']
        cited_id = row['cited_id']
        confirmation = str(row['confirmed']).strip().lower()
        
        if confirmation == 'yes':
            citation_matrix.loc[citing_id, cited_id] = 1
            confirmed_count += 1
        elif confirmation == 'no':
            # Ensure it's set to 0 (though it should already be)
            citation_matrix.loc[citing_id, cited_id] = 0
            rejected_count += 1
        else:
            skipped_count += 1
    
    # Print summary
    print(f"\nCitation Confirmation Summary:")
    print(f"- Confirmed: {confirmed_count}")
    print(f"- Rejected: {rejected_count}")
    print(f"- Skipped: {skipped_count}")
    
    # Save updated matrix
    citation_matrix.to_csv('../datasets/interconnections/citation_matrix.csv')
    print("Updated citation matrix saved to '../datasets/interconnections/citation_matrix.csv'")
    
    return citation_matrix

In [ ]:

# Execute the pipeline
citation_matrix, citation_details, uncertain_matches = build_citation_network_with_excel(
    df_id_bibtex, 
    pdf_path_dict, 
    path_constant_part
)

# Save initial citation matrix (with only high-confidence matches)
citation_matrix.to_csv('../datasets/interconnections/citation_matrix_initial.csv')

# Display instructions for the user
print("\n" + "="*80)
print("NEXT STEPS:")
print("1. Open the file '../datasets/interconnections/citation_confirmation.xlsx'")
print("2. For each row, review the potential citation")
print("3. In the 'confirmed' column, enter:")
print("   - 'Yes' if it's a valid citation")
print("   - 'No' if it's not a valid citation")
print("   - Leave blank to skip")
print("4. Save the file and run the code below to update the citation matrix:")
print("   citation_matrix = import_citation_confirmations(citation_matrix)")
print("="*80)



NEXT STEPS:
1. Open the file '../datasets/interconnections/citation_confirmation.xlsx'
2. For each row, review the potential citation
3. In the 'confirmed' column, enter:
   - 'Yes' if it's a valid citation
   - 'No' if it's not a valid citation
   - Leave blank to skip
4. Save the file and run the code below to update the citation matrix:
   citation_matrix = import_citation_confirmations(citation_matrix)


In [32]:
# Note: There is an error for the Paper 18 since the PDF seems not well-readable. It cites none of the other studies anyways.

In [33]:
# To run after manual confirmation:
citation_matrix = import_citation_confirmations(citation_matrix)

# Print summary statistics
num_citations = citation_matrix.sum().sum()
num_papers_with_citations = (citation_matrix.sum(axis=1) > 0).sum()
num_papers_cited = (citation_matrix.sum(axis=0) > 0).sum()

print(f"\nUpdated Citation Network Summary:")
print(f"Total citations between corpus papers: {num_citations}")
print(f"Papers that cite others in the corpus: {num_papers_with_citations}")
print(f"Papers that are cited by others: {num_papers_cited}")


Citation Confirmation Summary:
- Confirmed: 76
- Rejected: 107
- Skipped: 0
Updated citation matrix saved to '../datasets/interconnections/citation_matrix.csv'

Updated Citation Network Summary:
Total citations between corpus papers: 459
Papers that cite others in the corpus: 106
Papers that are cited by others: 102
